In [ ]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# ============================================================
# LOAD FILES
# ============================================================

df = pd.read_csv("../data/drivearabia_car_depreciation_valuation/original/suzuki.csv")
dep_df = pd.read_csv("../data/drivearabia_car_depreciation_valuation/annual_dep_rate.csv")

# ============================================================
# CLEAN DEPRECIATION DATA
# ============================================================

dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# ============================================================
# SUZUKI MODEL MAPPING
# ============================================================

MODEL_MAPPING = {
    "suzuki-baleno": "BALENO",
    "suzuki-celerio": "CELERIO",
    "suzuki-ciaz": "CIAZ",
    "suzuki-dzire": "DZIRE",
    "suzuki-ertiga": "ERTIGA",
    "suzuki-fronx": "FRONX",
    "suzuki-grand-vitara": "GRAND VITARA",
    "suzuki-ignis": "IGNIS",
    "suzuki-jimny": "JIMNY",
    "suzuki-s-presso": "S-PRESSO",
    "suzuki-swift": "SWIFT",
    "suzuki-vitara": "VITARA",
    "suzuki-xl7": "XL7",
}

df["make"] = "SUZUKI"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# ============================================================
# BUILD LOOKUP
# ============================================================

dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# ============================================================
# MERGE DEPRECIATION RATE
# ============================================================

df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# ============================================================
# CAR AGE
# ============================================================

df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# ============================================================
# DEPRECIATED VALUE
# ============================================================

def calc_depreciated_value(row):

    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# ============================================================
# VALIDATION
# ============================================================

print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

print("\nUnmatched Models:")
print(
    df[df["annual_dep_rate"].isna()]
    ["model_slug"]
    .drop_duplicates()
    .tolist()
)

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "../data/drivearabia_car_depreciation_valuation/depreciated/suzuki_dep.csv",
    index=False
)

print("\nSaved: ../data/drivearabia_car_depreciation_valuation/depreciated/suzuki_dep.csv")

Total Rows: 90
Matched Rates: 77
Missing Rates: 13

Unmatched Models:
['suzuki-jimny-5-door', 'suzuki-swift-dzire']

Saved: data/suzuki_dep.csv
